# Hult Form — Using `general_llm_app`

This notebook shows how to use the simplified `LLM` class to generate form questions from KPI definitions.

Compare this to `hult_form.ipynb` — same result, way less boilerplate.

In [ ]:
import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from llm import LLM

llm = LLM(model="gpt-4o-mini")

In [ ]:
# Load the KPI data
kpi_df = pd.read_csv("../data/Hult Prize - Form Questions - Sheet2.csv")
kpi_df.head()

In [ ]:
# Batch call: one LLM call per row, using the prompt template + schema
results = llm.batch(
    "tasks/hult_form/prompt.md",
    rows=kpi_df.to_dict("records"),
    variables_fn=lambda row: {
        "KPI_ID": str(row["KPI ID"]),
        "KPI_Definition": str(row["KPI Definition"]),
        "Calculation_Method_Examples": str(row["Calculation Method & Examples"]),
    },
    schema_path="tasks/hult_form/schema.json",
)

# Attach metadata from the original rows
for r, (_, row) in zip(results, kpi_df.iterrows()):
    r["kpi_id"] = row["KPI ID"]
    r["metric_group"] = row["Metric Group"]

In [ ]:
# Preview first result
print(json.dumps(results[0], indent=2))

In [ ]:
# Save JSON
output_path = "../output/hult/hult_form_questions.json"

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {output_path}")

In [ ]:
# Save Markdown
def result_to_markdown(r: dict) -> str:
    md = f"## {r['metric_human_name']} ({r['kpi_id']})\n\n"
    md += f"**Metric Group:** {r['metric_group']}\n\n"

    q1 = r["question_1_qualitative"]
    md += f"### Question 1: Definition & Narrative\n"
    md += f"**{q1['question_text']}**\n\n"
    md += f"*Helper:* {q1['helper_text']}\n\n"

    q2 = r["question_2_quantitative"]
    md += f"### Question 2: The Data Point\n"
    md += f"**{q2['question_text']}**\n\n"
    md += f"*Constraint:* {q2['constraint_note']}\n\n"
    md += f"*Unit:* {q2['unit']}\n\n"

    q3 = r["question_3_verification"]
    md += f"### Question 3: Verification\n"
    md += f"**{q3['question_text']}**\n\n"
    md += f"*Guidance:* {q3['guidance_text']}\n\n"

    md += "---\n\n"
    return md

markdown_content = "# Hult Prize - Generated Form Questions\n\n"
for r in results:
    markdown_content += result_to_markdown(r)

md_path = "../output/hult/hult_form_questions.md"
with open(md_path, "w") as f:
    f.write(markdown_content)

print(f"Saved to {md_path}")

In [ ]:
# Save CSV
def result_to_cell_text(r: dict) -> str:
    lines = []

    q1 = r["question_1_qualitative"]
    lines.append("Question No. 1: Definition & Narrative")
    lines.append(q1["question_text"])
    lines.append(f"_Helper: {q1['helper_text']}_")
    lines.append("Input Type: Text (Open-ended)")
    lines.append("")

    q2 = r["question_2_quantitative"]
    lines.append("Question No. 2: The Data Point")
    lines.append(q2["question_text"])
    lines.append(f"_Constraint: {q2['constraint_note']}_")
    lines.append(f"Input Type: Number ({q2['unit']})")
    lines.append("")

    q3 = r["question_3_verification"]
    lines.append("Question No. 3: Verification")
    lines.append(q3["question_text"])
    lines.append(f"_Guidance: {q3['guidance_text']}_")
    lines.append("Input Type: Text (Open-ended)")

    return "\n".join(lines)

csv_data = [{
    "KPI ID": r["kpi_id"],
    "Metric Group": r["metric_group"],
    "Metric Name": r["metric_human_name"],
    "Form Questions": result_to_cell_text(r)
} for r in results]

df_output = pd.DataFrame(csv_data)
csv_path = "../output/hult/hult_form_questions.csv"
df_output.to_csv(csv_path, index=False)

print(f"Saved to {csv_path}")
df_output.head()